# SCAPIS read-only DICOM inventory

Run the single code cell below. It reads DICOM headers only from `Q:\users\leejo\data\scapis\datahub` and writes all outputs only to `Q:\users\marfi\CT_analysis`.

**Organization**

- Folder name → site, CASC/CCTA protocol, and archive batch.
- DICOM files → patient, study, and `SeriesInstanceUID`.
- Series metadata → acquisition, reconstruction, geometry, contrast, dose, and cardiac/temporal parameters.
- Outputs → one workbook per site, a master 3D/4D workbook, CSV, and SQLite database.
- 4D is assigned conservatively only when multiple temporal positions or distinct matching cardiac phases are present.

In [ ]:
from pathlib import Path
import importlib.util
import os
import runpy
import subprocess
import sys

SOURCE = Path(r"Q:\users\leejo\data\scapis\datahub")  # READ ONLY
OUTPUT = Path(r"Q:\users\marfi\CT_analysis")           # ALL RESULTS
WORKERS = 8

if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source not found: {SOURCE}. Check that Q: is connected.")
source_abs = os.path.normcase(os.path.abspath(SOURCE))
output_abs = os.path.normcase(os.path.abspath(OUTPUT))
if os.path.commonpath([source_abs, output_abs]) == source_abs:
    raise RuntimeError("Output cannot be inside Leejo's read-only source folder.")

for module, package in [("pydicom", "pydicom>=2.4"), ("pandas", "pandas>=2.0"), ("openpyxl", "openpyxl>=3.1")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

candidates = [Path.cwd() / "04_build_site_dicom_inventories.py", Path.cwd() / "ct_analysis" / "04_build_site_dicom_inventories.py"]
ANALYZER = next((path for path in candidates if path.is_file()), None)
if ANALYZER is None:
    raise FileNotFoundError("Keep this notebook with 04_build_site_dicom_inventories.py, or run it from the repository root.")

OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"READ ONLY: {SOURCE}")
print(f"SAVE ONLY: {OUTPUT}")
sys.argv = [str(ANALYZER), "--root", str(SOURCE), "--output", str(OUTPUT), "--workers", str(WORKERS)]
try:
    runpy.run_path(str(ANALYZER), run_name="__main__")
except SystemExit as error:
    if error.code not in (None, 0):
        raise